In [ ]:
# py -v = 3.12.2
#https://github.com/Sprayer115/BI
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import re

In [ ]:
train = pd.read_csv("./train.csv") #Datensatz angeben
test = pd.read_csv("./test.csv") #Datensatz angeben

In [ ]:
# dropping irrelevant columns
train.drop('Cabin', axis=1, inplace=True)
train.drop('PassengerId', axis=1, inplace=True)
train.drop('Ticket', axis=1, inplace=True)
#train = train.dropna(subset=['Embarked'])
train.fillna(train['Embarked'].mode()[0])

# Replace Sex Values male=1, female=2
train['Sex'] = train['Sex'].str.lower().map({'male': 1, 'female': 2}).fillna(train['Sex'])# Count rows with NaN values in any column
train['Embarked'] = train['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})

train['Age'] = train['Age'].fillna(train['Age'].median())
#train = train.dropna(subset=['Age']) # Had no effect on the result 0,07 -> 0,08
train


In [ ]:
# Modify Dataset and Exploration

nan_count = train.isna().any(axis=1).sum()
# To identify which columns have NaN values and count them
nan_columns = train.isna().sum()
print(nan_columns)

In [ ]:
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
train['IsAlone'] = np.where(train['FamilySize'] == 1, 1, 0)

# Extract Title from Name
def get_title(name):
    title_search = re.search(r' ([A-Za-z]+)\.', name)  # Verwenden eines rohen String-Literals
    if title_search:
        return title_search.group(1)
    return ""

train['Title'] = train['Name'].apply(get_title)
train['Title'] = train['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
train['Title'] = train['Title'].replace('Mlle', 'Miss')
train['Title'] = train['Title'].replace('Ms', 'Miss')
train['Title'] = train['Title'].replace('Mme', 'Mrs')
train['Title'] = train['Title'].map({"Mr": 0, "Miss": 1, "Mrs": 2, "Master": 3, "Rare": 4})
train['Title'] = train['Title'].fillna(0)
train.drop('Name', axis=1, inplace=True)
train

In [ ]:
test.drop(['Cabin', 'PassengerId', 'Ticket'], axis=1, inplace=True)
test['Sex'] = test['Sex'].str.lower().map({'male': 1, 'female': 2}).fillna(test['Sex'])
test['Embarked'] = test['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})
test['Embarked'] = test['Embarked'].fillna(test['Embarked'].mode()[0])
test['Age'] = test['Age'].fillna(test['Age'].median())
test['Fare'] = test['Fare'].fillna(test['Fare'].median())  # Fehlende Werte in Fare auff?llen
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1
test['IsAlone'] = np.where(test['FamilySize'] == 1, 1, 0)

test['Title'] = test['Name'].apply(get_title)
test['Title'] = test['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
test['Title'] = test['Title'].replace('Mlle', 'Miss')
test['Title'] = test['Title'].replace('Ms', 'Miss')
test['Title'] = test['Title'].replace('Mme', 'Mrs')
test['Title'] = test['Title'].map({"Mr": 0, "Miss": 1, "Mrs": 2, "Master": 3, "Rare": 4})
test['Title'] = test['Title'].fillna(0)
test.drop('Name', axis=1, inplace=True)

In [ ]:
features = ["Sex", "Pclass", "Fare", "Age", "FamilySize", "Title"]
bestFeatures = ["Sex", "Pclass", "Fare", "Title", "IsAlone"]
x_train = train[bestFeatures]
y_train = train["Survived"]
# Splitting the dataset into training and testing sets
#x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_test = test[bestFeatures]
#y_test = test["Survived"]

# Checking the sizes of the datasets
print("x_train size:", x_train.shape)
print("x_test size:", x_test.shape)
print("y_train size:", y_train.shape)
#print("y_test size:", y_test.shape)

 



In [ ]:
def evaluate_model(y_test, y_pred):
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:

'''
best = 1
best_acc = 0
for i in range(1, 800):
    rf_model = RandomForestClassifier(n_estimators=i, random_state=42)
    rf_model.fit(x_train, y_train)
    rf_y_pred = rf_model.predict(x_test)
    if accuracy_score(y_test, rf_y_pred) > best_acc:
        best = i
        best_acc = accuracy_score(y_test, rf_y_pred)
        
rf_model = RandomForestClassifier(n_estimators=best, random_state=42)
rf_model.fit(x_train, y_train)
rf_y_pred = rf_model.predict(x_test)
print(best)
print("Random Forest Evaluation:")
evaluate_model(y_test, rf_y_pred)

# Hyperparameter-Tuning f?r Logistic Regression
lr_params = [
    {'penalty': ['l1'], 'C': [0.01, 0.1, 1, 10, 100], 'solver': ['liblinear', 'saga'], 'max_iter': [10000]},
    {'penalty': ['l2'], 'C': [0.01, 0.1, 1, 10, 100], 'solver': ['lbfgs', 'liblinear', 'saga'], 'max_iter': [10000]}
]
lr_grid = GridSearchCV(LogisticRegression(random_state=42), lr_params, cv=5)
lr_grid.fit(x_train, y_train)
lr_best = lr_grid.best_estimator_


# Hyperparameter-Tuning f?r Random Forest
rf_params = {
    'n_estimators': [100, 120, 150, 200, 300],
    'max_features': [None, 'sqrt', 'log2'],
    'max_depth': [4, 6, 8],
    'criterion': ['gini', 'entropy']
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=5)
rf_grid.fit(x_train, y_train)
rf_best = rf_grid.best_estimator_

# %%
# Applying best Logistic Regression model
lr_best.fit(x_train, y_train)
lr_y_pred = lr_best.predict(x_test)

# Applying best Random Forest model
rf_best.fit(x_train, y_train)
rf_y_pred = rf_best.predict(x_test)
'''

In [ ]:
lr_best = LogisticRegression(C=0.1, max_iter=10000, penalty='l2', solver='liblinear')
lr_best.fit(x_train, y_train)
lr_y_pred = lr_best.predict(x_test)

rf_best = RandomForestClassifier(criterion='entropy', max_depth=8, max_features=None, random_state=42, n_estimators=300)
rf_best.fit(x_train, y_train)
rf_y_pred = rf_best.predict(x_test)

#print("Best parameters for Logistic Regression:", lr_grid.best_params_)
#print("Best parameters for Random Forest:", rf_grid.best_params_)

# Evaluation of Logistic Regression
print("Best Logistic Regression Evaluation:")
#evaluate_model(y_test, lr_y_pred)

# Evaluation of Random Forest
print("Best Random Forest Evaluation:")
#evaluate_model(y_test, rf_y_pred)
print(rf_y_pred)

# Cross-validation Logistic Regression
cv_scores = cross_val_score(rf_best, x_train, y_train, cv=10)
print("Cross-validation scores:", cv_scores)
print("Mean cross-validation score:", cv_scores.mean())

# Interpretation der Cross-Validation-Ergebnisse
'''
Die Cross-Validation hilft, die Generalisierbarkeit des Modells zu bewerten. 
Ein Mean Cross-Validation Score von ca. 0.8 zeigt, dass das Modell in verschiedenen Datenaufteilungen eine konstante Leistung erbringt. 
Die Streuung der Scores gibt auch Aufschluss ?ber die Stabilit?t des Modells.
'''


In [ ]:
# Visual Explorative Data Analysis (EDA)
plt.figure(figsize=(10, 6))
sns.heatmap(train.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

# Plotting the distribution of 'Age'
plt.figure(figsize=(10, 6))
sns.histplot(train['Age'], kde=True)
plt.title('Distribution of Age')
plt.show()

# Visualizing the survival rate by sex
plt.figure(figsize=(10, 6))
sns.barplot(x='Sex', y='Survived', data=train)
plt.title('Survival Rate by Sex')
plt.show()

# Visualizing the survival rate by class
plt.figure(figsize=(10, 6))
sns.barplot(x='Pclass', y='Survived', data=train)
plt.title('Survival Rate by Class')
plt.show()
train.corr()

'''
Korrelation Matrix zeigt, dass die ?berlebensrate am st?rksten mit dem Geschlecht korreliert ist. au?erderm ist die Klasse und der Titel stark korreliert.

Die weiteren Plots zeigen die Verteilung des Alters, die ?berlebensrate nach Geschlecht und Klasse. Was die korrelationen best?tigt.
'''